In [151]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DataLoadExample") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

In [152]:
df = spark.read.csv("hdfs://namenode:8020/user/data/raw/ecommerce/2026/02/14/ecommerce_logs.csv", header=True, inferSchema=True)

# 데이터 확인
df.show()

+-------------------+-------+----------+-----------+----------+-----------+------+--------+
|          timestamp|user_id|session_id| event_type|product_id|   category| price|quantity|
+-------------------+-------+----------+-----------+----------+-----------+------+--------+
|2026-02-14 09:00:15|    101|     s1001|       view|      p001|electronics|299.99|       1|
|2026-02-14 09:01:30|    101|     s1001|add_to_cart|      p001|electronics|299.99|       1|
|2026-02-14 09:05:45|    102|     s1002|       view|      p002|   clothing| 49.99|       1|
|2026-02-14 09:10:20|    101|     s1001|   purchase|      p001|electronics|299.99|       1|
|2026-02-14 09:15:30|    103|     s1003|       view|      p003|      books| 19.99|       1|
|2026-02-14 09:20:10|    102|     s1002|add_to_cart|      p002|   clothing| 49.99|       2|
|2026-02-14 09:25:45|    104|     s1004|       view|      p001|electronics|299.99|       1|
|2026-02-14 09:30:20|    102|     s1002|   purchase|      p002|   clothing| 49.9

In [153]:
df.describe()

DataFrame[summary: string, timestamp: string, user_id: string, session_id: string, event_type: string, product_id: string, category: string, price: string, quantity: string]

In [154]:
from pyspark.sql import functions as F

# 매출(total)
df = df.withColumn('total', F.col('price') * F.col('quantity'))

event_df = df.filter(df['event_type'] =='purchase')

# 카테고리별 이벤트 수, 고유 사용자 수, 총 매출, 구매 건수 계산
event_df = df.groupBy('category').agg(
    F.count('category').alias('event_count'),
    F.countDistinct('user_id').alias('user_count'),
    F.sum(F.when(F.col('event_type') == 'purchase', F.col('total'))).alias('total_revenue'),
    F.count(F.when(F.col('event_type') == 'purchase', F.col('event_type'))).alias('order_count')).orderBy('total_revenue', ascending=False)

event_df = event_df.withColumn('total_revenue', F.col('total_revenue').cast('float'))
event_df = event_df.na.fill(value = 0)

event_df.show()

+-----------+-----------+----------+-------------+-----------+
|   category|event_count|user_count|total_revenue|order_count|
+-----------+-----------+----------+-------------+-----------+
|electronics|          6|         3|       299.99|          1|
|   clothing|          4|         2|        99.98|          1|
|      books|          3|         1|        59.97|          1|
|       home|          2|         1|          0.0|          0|
+-----------+-----------+----------+-------------+-----------+



In [155]:
# timestamp 형식 변환

df = df.withColumn('timestamp', F.to_timestamp('timestamp'))
df.describe()
df.show()

+-------------------+-------+----------+-----------+----------+-----------+------+--------+------+
|          timestamp|user_id|session_id| event_type|product_id|   category| price|quantity| total|
+-------------------+-------+----------+-----------+----------+-----------+------+--------+------+
|2026-02-14 09:00:15|    101|     s1001|       view|      p001|electronics|299.99|       1|299.99|
|2026-02-14 09:01:30|    101|     s1001|add_to_cart|      p001|electronics|299.99|       1|299.99|
|2026-02-14 09:05:45|    102|     s1002|       view|      p002|   clothing| 49.99|       1| 49.99|
|2026-02-14 09:10:20|    101|     s1001|   purchase|      p001|electronics|299.99|       1|299.99|
|2026-02-14 09:15:30|    103|     s1003|       view|      p003|      books| 19.99|       1| 19.99|
|2026-02-14 09:20:10|    102|     s1002|add_to_cart|      p002|   clothing| 49.99|       2| 99.98|
|2026-02-14 09:25:45|    104|     s1004|       view|      p001|electronics|299.99|       1|299.99|
|2026-02-1

In [156]:
# 시간대별 활동: 시간대(hour)별로 이벤트 수, 활성 사용자 수 계산

# 타임스탬프에서 시간대 추출
hourly_df = df.withColumn('timestamp', F.hour('timestamp'))

hourly_df = hourly_df.groupBy('timestamp').agg(
    F.count('event_type').alias('event_count'),
    # 시간대 별 고유 사용자 수로, 활성 사용자 수 계산
    countDistinct('user_id').alias('activated_user_count')).orderBy('timestamp')

hourly_df.show()

+---------+-----------+--------------------+
|timestamp|event_count|activated_user_count|
+---------+-----------+--------------------+
|        9|         13|                   6|
|       10|          2|                   2|
+---------+-----------+--------------------+



In [157]:
# 전환율 분석: 단계별 건수 계산

# 단계별 정렬위한 리스트 생성 .... > 어떻게?!
stage = ['view', 'add_to_cart', 'purchase']

stage_df = df.groupBy('event_type').agg(
    F.count('session_id').alias('stage_count')).orderBy('stage_count', ascending=False)
stage_df.show()

+-----------+-----------+
| event_type|stage_count|
+-----------+-----------+
|       view|          7|
|add_to_cart|          5|
|   purchase|          3|
+-----------+-----------+



In [158]:
# 사용자별 행동: 총 이벤트 수, 세션 수, 총 지출액 계산

user_df = df.groupBy('user_id').agg(
    F.count('event_type').alias('total_event_count'),
    F.countDistinct('session_id').alias('session_count'),
    F.sum(F.when(F.col('event_type') == 'purchase', F.col('total'))).alias('total_spent')).orderBy('user_id').limit(5)
    
user_df = user_df.withColumn('total_spent', F.col('total_spent').cast('float'))
user_df = user_df.na.fill(value = 0)
    
user_df.show()

+-------+-----------------+-------------+-----------+
|user_id|total_event_count|session_count|total_spent|
+-------+-----------------+-------------+-----------+
|    101|                3|            1|     299.99|
|    102|                3|            1|      99.98|
|    103|                3|            1|      59.97|
|    104|                2|            1|        0.0|
|    105|                2|            1|        0.0|
+-------+-----------------+-------------+-----------+



In [159]:
# 분석 결과 HDFS 저장

# 카테고리별 통계
event_df.write.parquet("hdfs://namenode:8020/user/data/processed/ecommerce/2026/02/14/category_stats", 
                        mode="overwrite")
# 시간대별 활동
hourly_df.write.parquet("hdfs://namenode:8020/user/data/processed/ecommerce/2026/02/14/hourly_activity", 
                        mode="overwrite")
# 전환율 분석
stage_df.write.parquet("hdfs://namenode:8020/user/data/processed/ecommerce/2026/02/14/funnel", 
                        mode="overwrite")
# 사용자별 행동
user_df.write.parquet("hdfs://namenode:8020/user/data/processed/ecommerce/2026/02/14/user_behavior", 
                        mode="overwrite")


In [160]:
# 저장된 결과 다시 읽기
spark.read.parquet("hdfs://namenode:8020/user/data/processed/ecommerce/2026/02/14/category_stats", header=True).show()
spark.read.parquet("hdfs://namenode:8020/user/data/processed/ecommerce/2026/02/14/hourly_activity", header=True).show()
spark.read.parquet("hdfs://namenode:8020/user/data/processed/ecommerce/2026/02/14/funnel", header=True).show()
spark.read.parquet("hdfs://namenode:8020/user/data/processed/ecommerce/2026/02/14/user_behavior", header=True).show()

+-----------+-----------+----------+-------------+-----------+
|   category|event_count|user_count|total_revenue|order_count|
+-----------+-----------+----------+-------------+-----------+
|electronics|          6|         3|       299.99|          1|
|   clothing|          4|         2|        99.98|          1|
|      books|          3|         1|        59.97|          1|
|       home|          2|         1|          0.0|          0|
+-----------+-----------+----------+-------------+-----------+

+---------+-----------+--------------------+
|timestamp|event_count|activated_user_count|
+---------+-----------+--------------------+
|        9|         13|                   6|
|       10|          2|                   2|
+---------+-----------+--------------------+

+-----------+-----------+
| event_type|stage_count|
+-----------+-----------+
|add_to_cart|          5|
|   purchase|          3|
|       view|          7|
+-----------+-----------+

+-------+-----------------+-------------+

In [161]:
from pyspark.sql import functions as F

def main():
    try:
        print("============================================================")
        print("E-COMMERCE DAILY REPORT - 2026-02-14")
        print("============================================================")

        df = spark.read.csv("hdfs://namenode:8020/user/data/raw/ecommerce/2026/02/14/ecommerce_logs.csv", header=True, inferSchema=True)

        # 매출(total)
        df = df.withColumn('total', F.col('price') * F.col('quantity'))

        print("[1] Category Performance")
        event_df = df.filter(df['event_type'] =='purchase')

        # 카테고리별 이벤트 수, 고유 사용자 수, 총 매출, 구매 건수 계산
        event_df = df.groupBy('category').agg(
            F.count('category').alias('event_count'),
            F.countDistinct('user_id').alias('user_count'),
            F.sum(F.when(F.col('event_type') == 'purchase', F.col('total'))).alias('total_revenue'),
            F.count(F.when(F.col('event_type') == 'purchase', F.col('event_type'))).alias('order_count')).orderBy('total_revenue', ascending=False)

        event_df = event_df.withColumn('total_revenue', F.col('total_revenue').cast('float'))
        event_df = event_df.na.fill(value = 0)

        event_df.show()

        
        print("[2] Hourly Activity")
        df = df.withColumn('timestamp', F.to_timestamp('timestamp'))
        
        # 시간대별 활동: 시간대(hour)별로 이벤트 수, 활성 사용자 수 계산

        # 타임스탬프에서 시간대 추출
        hourly_df = df.withColumn('timestamp', F.hour('timestamp'))

        hourly_df = hourly_df.groupBy('timestamp').agg(
            F.count('event_type').alias('event_count'),
            # 시간대 별 고유 사용자 수로, 활성 사용자 수 계산
            countDistinct('user_id').alias('activated_user_count')).orderBy('timestamp')
        hourly_df.show()
        
        
        print("[3] Event Funnel")
        # 전환율 분석: 단계별 건수 계산

        # 단계별 정렬위한 리스트 생성 .... > 어떻게?!
        stage = ['view', 'add_to_cart', 'purchase']

        stage_df = df.groupBy('event_type').agg(
            F.count('session_id').alias('stage_count')).orderBy('stage_count', ascending=False)
        stage_df.show()
        
        
        print("[4] Top 5 Users by Revenue")
        # 사용자별 행동: 총 이벤트 수, 세션 수, 총 지출액 계산

        user_df = df.groupBy('user_id').agg(
            F.count('event_type').alias('total_event_count'),
            F.count('session_id').alias('session_count'),
            F.sum(F.when(F.col('event_type') == 'purchase', F.col('total'))).alias('total_spent')).orderBy('user_id').limit(5)
        user_df.show()
        
        # 분석 결과 HDFS 저장

        # 카테고리별 통계
        event_df.write.parquet("hdfs://namenode:8020/user/data/processed/ecommerce/2026/02/14/category_stats", 
                                mode="overwrite")
        # 시간대별 활동
        hourly_df.write.parquet("hdfs://namenode:8020/user/data/processed/ecommerce/2026/02/14/hourly_activity", 
                                mode="overwrite")
        # 전환율 분석
        stage_df.write.parquet("hdfs://namenode:8020/user/data/processed/ecommerce/2026/02/14/funnel", 
                                mode="overwrite")
        # 사용자별 행동
        user_df.write.parquet("hdfs://namenode:8020/user/data/processed/ecommerce/2026/02/14/user_behavior", 
                                mode="overwrite")
        print("모든 결과 저장")
        
    except Exception as e:
        logger.error(f"❌ Pipeline failed: {e}")
        import traceback
        traceback.print_exc()
        raise

In [162]:
main()

E-COMMERCE DAILY REPORT - 2026-02-14
[1] Category Performance


+-----------+-----------+----------+-------------+-----------+
|   category|event_count|user_count|total_revenue|order_count|
+-----------+-----------+----------+-------------+-----------+
|electronics|          6|         3|       299.99|          1|
|   clothing|          4|         2|        99.98|          1|
|      books|          3|         1|        59.97|          1|
|       home|          2|         1|          0.0|          0|
+-----------+-----------+----------+-------------+-----------+

[2] Hourly Activity


+---------+-----------+--------------------+
|timestamp|event_count|activated_user_count|
+---------+-----------+--------------------+
|        9|         13|                   6|
|       10|          2|                   2|
+---------+-----------+--------------------+

[3] Event Funnel


+-----------+-----------+
| event_type|stage_count|
+-----------+-----------+
|       view|          7|
|add_to_cart|          5|
|   purchase|          3|
+-----------+-----------+

[4] Top 5 Users by Revenue


+-------+-----------------+-------------+-----------+
|user_id|total_event_count|session_count|total_spent|
+-------+-----------------+-------------+-----------+
|    101|                3|            3|     299.99|
|    102|                3|            3|      99.98|
|    103|                3|            3|      59.97|
|    104|                2|            2|       null|
|    105|                2|            2|       null|
+-------+-----------------+-------------+-----------+



모든 결과 저장
